# Aluminium Beverage Can LCA

Cradle-to-grave carbon footprint of one 0.5 L aluminium beverage can, Brightway for the calculation, BAFU for the background. Same functional unit (0.5 L container) as the PET and glass bottle notebooks, so all three line up for comparison.

This one mixes confirmed picks with search-then-fill ones. A couple of BAFU activities already turned up in earlier searches during the PET bottle work, specifically `'Disposal, aluminium, 0% water, to municipal incineration'` and `'Disposal, aluminium, 0% water, to sanitary landfill'`, so those are hard-coded here with confidence. Aluminium production (primary versus secondary/recycled) and can-forming hadn't been searched yet, so those sections follow the same search-then-fill pattern as the t-shirt notebook: run the search, read the real result, then assign the exact index.

Primary versus secondary aluminium matters a lot here. Producing aluminium from bauxite ore (primary, via electrolysis) takes roughly twenty times more energy than remelting scrap (secondary). Beverage cans typically already contain a high share of recycled metal, so the material stage below is modelled as a mix of the two rather than one generic aluminium activity, a single figure either way would misrepresent the real footprint.

What's in here:
1. Project setup
2. Import BAFU (ecoSpold1), same debugged procedure as the other notebooks
3. Search BAFU for aluminium production and can-forming
4. Recycled content and end-of-life assumptions
5. Build the foreground activity
6. Run the LCA
7. Contribution breakdown by stage
8. Monte Carlo

## 1. Project setup

In [ ]:
import brightway2 as bw

bw.projects.set_current("Aluminium Can LCA")
bw.bw2setup()

print(list(bw.databases))

Biosphere database already present!!! No setup is needed
['biosphere3', 'BAFU', 'Aluminium Can']


In [2]:
import brightway2 as bw

bw.projects.set_current("Aluminium Can LCA")

act = bw.get_activity(("Aluminium Can", "aluminium_can_0.5L"))
act["reference product"] = act["name"]
act.save()

print(act["reference product"])

Aluminium can, 0.5 litre, cradle-to-grave


## 2. Import BAFU (ecoSpold1)

Same procedure worked out in the PET bottle notebook. Update the path below to the extracted ecoSpold1 folder.

In [3]:
import bw2io as bi

ECOSPOLD_PATH = "/Users/abhi/ecoSpold files"

importer = bi.SingleOutputEcospold1Importer(ECOSPOLD_PATH, "BAFU")

Extracting XML data from 11947 datasets


/opt/anaconda3/envs/ab/lib/python3.11/site-packages/bw2io/extractors/ecospold1.py:320: RuntimeWarning: divide by zero encountered in log
  "loc": np.log(np.abs(mean)),
/opt/anaconda3/envs/ab/lib/python3.11/site-packages/bw2io/extractors/ecospold1.py:320: RuntimeWarning: divide by zero encountered in log
  "loc": np.log(np.abs(mean)),
/opt/anaconda3/envs/ab/lib/python3.11/site-packages/bw2io/extractors/ecospold1.py:320: RuntimeWarning: divide by zero encountered in log
  "loc": np.log(np.abs(mean)),
/opt/anaconda3/envs/ab/lib/python3.11/site-packages/bw2io/extractors/ecospold1.py:320: RuntimeWarning: divide by zero encountered in log
  "loc": np.log(np.abs(mean)),
/opt/anaconda3/envs/ab/lib/python3.11/site-packages/bw2io/extractors/ecospold1.py:320: RuntimeWarning: divide by zero encountered in log
  "loc": np.log(np.abs(mean)),
/opt/anaconda3/envs/ab/lib/python3.11/site-packages/bw2io/extractors/ecospold1.py:320: RuntimeWarning: divide by zero encountered in log
  "loc": np.log(np.abs(

Extracted 11947 datasets in 6.97 seconds


In [4]:
fields_to_clean = ["name", "categories", "unit", "reference product", "location"]

for ds in importer.data:
    for field in fields_to_clean:
        if ds.get(field) is None:
            ds[field] = ""
    for exc in ds.get("exchanges", []):
        for field in fields_to_clean:
            if exc.get(field) is None:
                exc[field] = ""

In [5]:
importer.apply_strategies()
importer.match_database("biosphere3", fields=["name", "categories"])
importer.statistics()

Applying strategy: normalize_units
Applying strategy: assign_only_product_as_production
Applying strategy: clean_integer_codes
Applying strategy: drop_unspecified_subcategories
Applying strategy: normalize_biosphere_categories
Applying strategy: normalize_biosphere_names
Applying strategy: strip_biosphere_exc_locations
Applying strategy: update_ecoinvent_locations
Applying strategy: set_code_by_activity_hash
Applying strategy: link_iterable_by_fields
Applying strategy: link_technosphere_by_activity_hash
Applied 11 strategies in 8.30 seconds
Applying strategy: link_iterable_by_fields
11947 datasets
420063 exchanges
295213 unlinked exchanges
  Type biosphere: 2679 unique unlinked exchanges
  Type technosphere: 24 unique unlinked exchanges


(11947, 420063, 295213)

In [6]:
CATEGORY_FIX = {
    ("emissions to air",): ("air",),
    ("emissions to air", "high. pop."): ("air", "urban air close to ground"),
    ("emissions to air", "low. pop."): ("air", "non-urban air or from high stacks"),
    ("emissions to air", "stratosphere + troposphere"): ("air", "lower stratosphere + upper troposphere"),
    ("emissions to soil",): ("soil",),
    ("emissions to soil", "agricultural"): ("soil", "agricultural"),
    ("emissions to soil", "industrial"): ("soil", "industrial"),
    ("emissions to water",): ("water",),
    ("emissions to water", "ground-"): ("water", "ground-"),
    ("emissions to water", "ocean"): ("water", "ocean"),
    ("emissions to water", "river"): ("water", "surface water"),
    ("resources",): ("natural resource",),
    ("resources", "land"): ("natural resource", "land"),
    ("resources", "in ground"): ("natural resource", "in ground"),
    ("resources", "in air"): ("natural resource", "in air"),
    ("resources", "in water"): ("natural resource", "in water"),
    ("resources", "biotic"): ("natural resource", "biotic"),
}

fixed_count = 0
for ds in importer.data:
    for exc in ds.get("exchanges", []):
        if exc.get("type") == "biosphere":
            cats = tuple(exc.get("categories") or ())
            if cats in CATEGORY_FIX:
                exc["categories"] = CATEGORY_FIX[cats]
                fixed_count += 1

print(f"Rewrote categories on {fixed_count} exchanges")

Rewrote categories on 268970 exchanges


In [7]:
importer.match_database("biosphere3", fields=["name", "categories"])
importer.statistics()

Applying strategy: link_iterable_by_fields
11947 datasets
420063 exchanges
114060 unlinked exchanges
  Type biosphere: 1155 unique unlinked exchanges
  Type technosphere: 24 unique unlinked exchanges


(11947, 420063, 114060)

In [8]:
unlinked = list(importer.unlinked)
bio_unlinked = [u for u in unlinked if u.get("type") == "biosphere"]
tech_unlinked = [u for u in unlinked if u.get("type") == "technosphere"]

print(f"{len(bio_unlinked)} unique unlinked biosphere flows")
for u in bio_unlinked[:20]:
    print(u.get("name"), "|", u.get("categories"), "|", u.get("unit"))

print(f"\n{len(tech_unlinked)} unique unlinked technosphere")
for u in tech_unlinked[:24]:
    print(u.get("name"), "|", u.get("location"), "|", u.get("unit"))

1155 unique unlinked biosphere flows
NMVOC, non-methane volatile organic compounds, unspecified origin | ('air', 'non-urban air or from high stacks') | kilogram
Mercury | ('air', 'non-urban air or from high stacks') | kilogram
Occupation, industrial area, built up | ('natural resource', 'land') | square meter-year
Transformation, to industrial area, built up | ('natural resource', 'land') | square meter
Particulates, < 2.5 um | ('air', 'urban air close to ground') | kilogram
Copper | ('air', 'urban air close to ground') | kilogram
Particulates, < 2.5 um | ('air', 'non-urban air or from high stacks') | kilogram
Mercury | ('air', 'urban air close to ground') | kilogram
Selenium | ('air', 'non-urban air or from high stacks') | kilogram
Selenium | ('air', 'urban air close to ground') | kilogram
Nickel | ('air', 'non-urban air or from high stacks') | kilogram
Zinc | ('air', 'non-urban air or from high stacks') | kilogram
Nickel | ('air', 'urban air close to ground') | kilogram
Zinc | ('air'

In [9]:
importer.drop_unlinked(i_am_reckless=True)
importer.statistics()

Applying strategy: drop_unlinked
Applied 1 strategies in 0.07 seconds
11947 datasets
306003 exchanges
0 unlinked exchanges
  


(11947, 306003, 0)

In [11]:
importer.write_database()
print(list(bw.databases))

Title: Writing activities to SQLite3 database:
  Started: 08/06/2026 11:10:59
  Finished: 08/06/2026 11:11:07
  Total time elapsed: 00:00:07
  CPU %: 96.50
  Memory %: 7.00
Created database: BAFU
['biosphere3', 'BAFU', 'Aluminium Can']


## 3. Search BAFU for aluminium production and can-forming

Run each search and look at the real results before picking anything.

In [12]:
ei = bw.Database("BAFU")

# Primary aluminium, produced from bauxite ore via electrolysis
for r in ei.search("aluminium production"):
    print(r, "|", r["location"])

print("---")
for r in ei.search("aluminium primary"):
    print(r, "|", r["location"])

'Aluminium, production mix for aluminium profiles, SZFF 2014, at plant' (kilogram, CH, ['metals', 'extraction']) | CH
'Aluminium, production mix, at plant' (kilogram, RER, ['metals', 'extraction']) | RER
'Gallium, in Bayer liquor from aluminium production, at plant' (kilogram, GLO, ['metals', 'non ferro']) | GLO
'Aluminium, production mix, cast alloy, at plant' (kilogram, RER, ['metals', 'extraction']) | RER
'Aluminium, production mix, wrought alloy, at plant' (kilogram, RER, ['metals', 'extraction']) | RER
'Aluminium collector foil production, for Li-ion battery' (kilogram, GLO, ['electronics', 'batteries']) | GLO
'Aluminium profile, uncoated, secondary production (100% Rec.)' (kilogram, CH, ['construction materials', 'others']) | CH
'Aluminium sheet, uncoated, secondary production (100% Rec.)' (kilogram, CH, ['construction materials', 'others']) | CH
'Sealing sheet, aluminium, secondary production (100% Rec.)' (kilogram, CH, ['construction materials', 'others']) | CH
'xx Recycling al

In [13]:
# Secondary aluminium, remelted from scrap, much lower energy use than primary
for r in ei.search("aluminium secondary"):
    print(r, "|", r["location"])

print("---")
for r in ei.search("aluminium scrap"):
    print(r, "|", r["location"])

print("---")
for r in ei.search("aluminium recycling"):
    print(r, "|", r["location"])

'Aluminium, secondary, from old scrap, at plant' (kilogram, RER, ['metals', 'extraction']) | RER
'Aluminium, secondary, from new scrap, at plant' (kilogram, RER, ['metals', 'extraction']) | RER
'Aluminium profile, uncoated, secondary production (100% Rec.)' (kilogram, CH, ['construction materials', 'others']) | CH
'Aluminium sheet, uncoated, secondary production (100% Rec.)' (kilogram, CH, ['construction materials', 'others']) | CH
'Sealing sheet, aluminium, secondary production (100% Rec.)' (kilogram, CH, ['construction materials', 'others']) | CH
'Aluminium, production mix for aluminium profiles, SZFF 2014, at plant' (kilogram, CH, ['metals', 'extraction']) | CH
'Aluminium scrap, new, at plant' (kilogram, RER, ['metals', 'waste metals']) | RER
'Aluminium, production mix, at plant' (kilogram, RER, ['metals', 'extraction']) | RER
'Aluminium, production mix, cast alloy, at plant' (kilogram, RER, ['metals', 'extraction']) | RER
'Aluminium, production mix, wrought alloy, at plant' (kilogr

In [14]:
# Sheet rolling and can forming. BAFU may not have a can-specific process,
# in which case a general aluminium sheet or rolling process is the closest proxy.
for r in ei.search("aluminium sheet"):
    print(r, "|", r["location"])

print("---")
for r in ei.search("rolling"):
    print(r, "|", r["location"])

print("---")
for r in ei.search("impact extrusion"):
    print(r, "|", r["location"])

'Sheet rolling, aluminium' (kilogram, RER, ['metals', 'chipless shaping']) | RER
'Powder coating, aluminium sheet' (square meter, RER, ['metals', 'coating']) | RER
'Aluminium sheet, uncoated' (kilogram, CH, ['construction materials', 'others']) | CH
'Anodising, aluminium sheet' (square meter, RER, ['metals', 'coating']) | RER
'Selective coating, aluminium sheet, nickel pigmented aluminium oxide' (square meter, SK, ['metals', 'coating']) | SK
'Aluminium sheet, uncoated, with resource correction' (kilogram, CH, ['construction materials', 'others']) | CH
'Mineral wool insulation 60mm, cladding with aluminium sheet' (square meter, CH, ['heating', 'production of components']) | CH
'Mineral wool insulation 100mm, cladding with aluminium sheet' (square meter, CH, ['heating', 'production of components']) | CH
'Mineral wool insulation 50mm, cladding with aluminium sheet' (square meter, CH, ['heating', 'production of components']) | CH
'Aluminium sheet, uncoated, secondary production (100% Rec.)

Fill in the picks below once the searches above have been run and the correct index is known for each. End-of-life and transport are already confirmed from earlier BAFU searches (the same disposal and transport searches from the PET bottle model), so those go in directly.

In [ ]:

# Confirmed from earlier BAFU searches, same picks used in the PET bottle model
truck = ei.search("transport lorry")[2]
print(truck)  # 'Transport, freight, lorry, fleet average' (ton kilometer, CH)

incineration = ei.search("municipal solid waste incineration")[12]
print(incineration)  # 'Disposal, aluminium, 0% water, to municipal incineration' (CH)

landfill = ei.search("sanitary landfill")[1]
print(landfill)  # 'Disposal, aluminium, 0% water, to sanitary landfill' (CH)

'Transport, freight, lorry, fleet average' (ton kilometer, CH, ['transport systems', 'road\\Transformation'])
'Disposal, aluminium, 0% water, to municipal incineration' (kilogram, CH, ['waste management', 'municipal incineration'])
'Disposal, paint, 0% water, to sanitary landfill' (kilogram, CH, ['landfill', 'sanitary landfill'])


Check the two printed lines above against the comments before moving on. Index positions can shift slightly between BAFU versions, so confirm rather than assume.

## 4. Recycled content and end-of-life assumptions

The aluminium can industry reports a high average recycled content, commonly cited around 68%, though it varies by region and should be adjusted to whatever market is actually being studied. The material stage below is split between primary and secondary aluminium according to that share, rather than lumping it all under one generic aluminium activity.

Aluminium cans also have one of the highest collection and recycling rates of any packaging format, since the metal keeps high scrap value, which is why the end-of-life split below leans toward recycling. These figures are illustrative and should be swapped for a rate specific to the region being studied.

In [16]:
# ---- Physical parameters ----
CAN_MASS_KG = 0.018   # a 0.5 L aluminium can, body and lid combined, weighs about 18 g
DIST_KM = 100          # factory to retailer distance, same assumption as the other notebooks

# ---- Recycled content of the incoming aluminium ----
RECYCLED_CONTENT = 0.68   # share of the can's aluminium that is secondary (recycled) metal

# ---- End-of-life split ----
EoL_RECYCLE = 0.68     # collected and recycled, mirrors the recycled content share above
EoL_INCIN = 0.20
EoL_LANDFILL = 0.12

print("Assumptions set.")

Assumptions set.


## 5. Build the foreground activity

This stays a simplified model until the picks in section 3 are confirmed. Right now the recycling share at end-of-life is sent through the same incineration and landfill split as the non-recycled share, since there's no confirmed BAFU aluminium recycling process yet. Once `secondary_aluminium` is confirmed, the placeholder line below should route the recycled share there instead, that's the more accurate way to model an open-loop recycling credit.

In [18]:
DB_NAME = "Aluminium Can"

if DB_NAME in bw.databases:
    del bw.databases[DB_NAME]

exchanges = [
    {"input": (DB_NAME, "aluminium_can_0.5L"), "amount": 1, "unit": "piece", "type": "production"},

    # ---- Material stage, blend of primary and secondary aluminium ----
    # Uncomment once the picks from section 3 are confirmed:
    # {"input": primary_aluminium.key, "amount": CAN_MASS_KG * (1 - RECYCLED_CONTENT), "unit": "kilogram", "type": "technosphere"},
    # {"input": secondary_aluminium.key, "amount": CAN_MASS_KG * RECYCLED_CONTENT, "unit": "kilogram", "type": "technosphere"},

    # ---- Can forming (sheet rolling, stamping) ----
    # {"input": can_forming.key, "amount": CAN_MASS_KG, "unit": "kilogram", "type": "technosphere"},

    # ---- Distribution ----
    {"input": truck.key, "amount": CAN_MASS_KG * DIST_KM / 1000, "unit": "ton kilometer", "type": "technosphere"},

    # ---- End of life, confirmed BAFU picks ----
    {"input": incineration.key, "amount": CAN_MASS_KG * EoL_INCIN, "unit": "kilogram", "type": "technosphere"},
    {"input": landfill.key, "amount": CAN_MASS_KG * EoL_LANDFILL, "unit": "kilogram", "type": "technosphere"},
    # The EoL_RECYCLE share is not yet linked to a recycling process, see the note above.
]

foreground = bw.Database(DB_NAME)
foreground.write({
    (DB_NAME, "aluminium_can_0.5L"): {
        "name": "Aluminium can, 0.5 litre, cradle-to-grave",
        "reference product": "Aluminium can, 0.5 litre, cradle-to-grave",
        "unit": "piece",
        "location": "CH",
        "type": "process",
        "exchanges": exchanges,
    }
})

print("Database written:", list(bw.databases))

Title: Writing activities to SQLite3 database:
  Started: 08/06/2026 11:12:35
  Finished: 08/06/2026 11:12:35
  Total time elapsed: 00:00:00
  CPU %: 100.40
  Memory %: 0.68
Database written: ['biosphere3', 'BAFU', 'Aluminium Can']


Note to self: until the material and forming picks are uncommented, this model only accounts for transport and part of the end-of-life. The score in section 6 will be far too low until the material stage is filled in, aluminium production is normally the dominant contributor to a can's footprint.

## 6. Run the LCA

Same method as the other notebooks, so all products can be compared on equal terms.

In [19]:
can = bw.get_activity((DB_NAME, "aluminium_can_0.5L"))

method = ("IPCC 2013", "climate change", "global warming potential (GWP100)")

lca = bw.LCA({can: 1}, method)
lca.lci()
lca.lcia()

print(f"GWP 100a: {lca.score:.4f} kg CO2-eq per can")

GWP 100a: 0.0006 kg CO2-eq per can


## 7. Monte Carlo uncertainty analysis

The single `GWP 100a` score above is one point estimate, it hides how much the result could vary given real uncertainty in every exchange behind it. Monte Carlo reruns the calculation many times, resampling every exchange from its uncertainty distribution each time, and returns a spread instead of one number. A mean plus a confidence interval is more honest to report than a single figure, especially when comparing products.

Two things to know first: `bw2calc`'s Monte Carlo solver can raise `ValueError: 'scipy.sparse.linalg.cgs' called with invalid atol='legacy'` on newer `scipy` versions, if that happens, run `conda install "scipy<1.12"` in the `ab` environment and restart the kernel. And Monte Carlo only produces a real spread if the exchanges being sampled actually carry uncertainty data, background BAFU exchanges might carry some from the original ecoSpold1 data, but the foreground exchanges written here (the can's own amounts) don't have uncertainty set by default, so they stay fixed across iterations unless it's added explicitly, for example through Activity Browser's Uncertainty tab.

In [20]:
method = ("IPCC 2013", "climate change", "global warming potential (GWP100)")

mc = bw.MonteCarloLCA({can: 1}, method)
n_iterations = 1000
scores = [next(mc) for _ in range(n_iterations)]

import numpy as np

print(f"Iterations: {n_iterations}")
print(f"Mean:   {np.mean(scores):.4f} kg CO2-eq per can")
print(f"Median: {np.median(scores):.4f} kg CO2-eq per can")
print(f"Std dev: {np.std(scores):.4f}")
print(f"5th to 95th percentile: {np.percentile(scores, 5):.4f} to {np.percentile(scores, 95):.4f} kg CO2-eq")

Iterations: 1000
Mean:   0.0006 kg CO2-eq per can
Median: 0.0006 kg CO2-eq per can
Std dev: 0.0001
5th to 95th percentile: 0.0005 to 0.0009 kg CO2-eq
